# 7 · Putting it together — the paper's 2×2

**Math primer — notebook 7 of 7.**

Every piece is now on the table. This notebook assembles them into Furusawa's
derivation, symbolically, and checks each step against `orthomount`.

**You should come out knowing:** how Eqs. (16), (22), (25), (27) and (29) follow
from the four preceding notebooks, and which symbol in the paper corresponds to
which line of code.

In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.linalg import eigh
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import primer
from primer import SERIES, INK, INK2, GRID, tidy
import orthomount as om
primer.use_style()
D, Dg = np.deg2rad, np.rad2deg

## 7.1 · The reduction, in one place

From **notebook 5**: a rigid body with its in-plane principal axes tilted by
$\alpha$ has

$$[M]_{\phi\psi}=\begin{bmatrix}I_x&-I_{zx}\\-I_{zx}&I_z\end{bmatrix}$$

From **notebook 6**: laterally symmetric mounts give a $[K]$ that couples only
$\phi$ with $\psi$ (and $x$ with $z$), and with the elastic principal axes tilted
by $\beta$,

$$[K]_{\phi\psi}=\begin{bmatrix}K_{\phi\phi}&K_{\phi\psi}\\K_{\phi\psi}&K_{\psi\psi}\end{bmatrix}$$

From **notebook 4**: with a pure $\phi$ excitation, mode 2 is silent iff
$M_{12}/M_{22}=K_{12}/K_{22}$.

Put those together and there is nothing left to derive — only algebra to carry
out. Let SymPy do it.

In [2]:
al, be = sp.symbols("alpha beta", real=True)
Ixi, Ieta, Izeta = sp.symbols("I_xi I_eta I_zeta", positive=True)
Kxi, Keta, Kzeta = sp.symbols("K_xi K_eta K_zeta", positive=True)

def Ry(a):
    return sp.Matrix([[sp.cos(a), 0, sp.sin(a)],
                      [0, 1, 0],
                      [-sp.sin(a), 0, sp.cos(a)]])

J = sp.simplify(Ry(al) * sp.diag(Ixi, Ieta, Izeta) * Ry(al).T)
Kb = sp.simplify(Ry(be) * sp.diag(Kxi, Keta, Kzeta) * Ry(be).T)

# the phi-psi 2x2 sub-blocks (rows/cols 0 and 2 of the 3x3 rotational blocks)
Mr = sp.Matrix([[J[0, 0], J[0, 2]], [J[2, 0], J[2, 2]]])
Kr = sp.Matrix([[Kb[0, 0], Kb[0, 2]], [Kb[2, 0], Kb[2, 2]]])

print("M_r =")
sp.pprint(sp.simplify(Mr))
print("\nK_r =")
sp.pprint(sp.simplify(Kr))

M_r =
⎡       2             2       (-Iₓᵢ + I_ζ)⋅sin(2⋅α)  ⎤
⎢Iₓᵢ⋅cos (α) + I_ζ⋅sin (α)    ─────────────────────  ⎥
⎢                                       2            ⎥
⎢                                                    ⎥
⎢  (-Iₓᵢ + I_ζ)⋅sin(2⋅α)           2             2   ⎥
⎢  ─────────────────────    Iₓᵢ⋅sin (α) + I_ζ⋅cos (α)⎥
⎣            2                                       ⎦

K_r =
⎡       2             2       (-Kₓᵢ + K_ζ)⋅sin(2⋅β)  ⎤
⎢Kₓᵢ⋅cos (β) + K_ζ⋅sin (β)    ─────────────────────  ⎥
⎢                                       2            ⎥
⎢                                                    ⎥
⎢  (-Kₓᵢ + K_ζ)⋅sin(2⋅β)           2             2   ⎥
⎢  ─────────────────────    Kₓᵢ⋅sin (β) + K_ζ⋅cos (β)⎥
⎣            2                                       ⎦


## 7.2 · Eq. (16), symbolically

The silencing condition from notebook 4 is $M_{12}/M_{22}=K_{12}/K_{22}$.
Define $R$ as the paper does — remembering $I_{zx}=-J_{xz}$, so
$M_{12}/M_{22}=-I_{zx}/I_z$:

$$R\equiv\frac{I_{zx}}{I_z}=-\frac{M_{12}}{M_{22}},\qquad
R=-\frac{K_{\phi\psi}}{K_{\psi\psi}}=-\frac{K_{12}}{K_{22}}$$

In [3]:
R_inertia  = sp.simplify(-Mr[0, 1] / Mr[1, 1])
R_stiff    = sp.simplify(-Kr[0, 1] / Kr[1, 1])
print("R from inertia   =", sp.simplify(R_inertia))
print("R from stiffness =", sp.simplify(R_stiff))

R from inertia   = (I_xi/2 - I_zeta/2)*sin(2*alpha)/(I_xi*sin(alpha)**2 + I_zeta*cos(alpha)**2)
R from stiffness = (K_xi/2 - K_zeta/2)*sin(2*beta)/(K_xi*sin(beta)**2 + K_zeta*cos(beta)**2)


## 7.3 · Eq. (22): introduce the ratios $p$ and $q$

Divide numerator and denominator through by $I_\zeta\cos^2\alpha$ (and
$K_\zeta\cos^2\beta$), and write $p=I_\xi/I_\zeta$, $q=K_\xi/K_\zeta$:

$$\frac{(p-1)\tan\alpha}{p\tan^2\alpha+1}=R=\frac{(q-1)\tan\beta}{q\tan^2\beta+1}$$

In [4]:
p, q, t = sp.symbols("p q t", positive=True)

# Substituting alpha = atan(t) turns every sin/cos into a rational function of
# t = tan(alpha), which is the form Eq. (22) is written in -- and which SymPy
# can then cancel to zero without any coaxing.
expr_i = sp.simplify(R_inertia.subs(Ixi, p*Izeta))
target = (p - 1)*sp.tan(al) / (p*sp.tan(al)**2 + 1)
diff_i = sp.simplify((expr_i - target).subs(al, sp.atan(t)))
print("R (inertia side) as a function of t = tan(alpha):")
sp.pprint(sp.simplify(expr_i.subs(al, sp.atan(t))))
print("\ninertia side minus Eq.(22) form:", diff_i)

expr_k = sp.simplify(R_stiff.subs(Kxi, q*Kzeta))
target_k = (q - 1)*sp.tan(be) / (q*sp.tan(be)**2 + 1)
diff_k = sp.simplify((expr_k - target_k).subs(be, sp.atan(t)))
print("stiffness side minus Eq.(22) form:", diff_k)

R (inertia side) as a function of t = tan(alpha):
t⋅(p - 1)
─────────
   2     
p⋅t  + 1 

inertia side minus Eq.(22) form: 0
stiffness side minus Eq.(22) form: 0


Both differences are zero, so Eq. (22) is confirmed. Note that the two sides are
*identical functions* of their arguments — which is why the paper's Fig. 2 can be
read either way.

## 7.4 · Eqs. (25) and (27): modal inertia and frequency

The surviving mode's shape is $[1,\ R]^\mathsf{T}$ (notebook 4: the ratio comes
straight from the orthogonality row). Its modal inertia is
$m_1=\{u^1\}^\mathsf{T}[M]_{\phi\psi}\{u^1\}$, and its frequency is
$\omega_1^2=k_1/m_1$.

In [5]:
u1 = sp.Matrix([1, sp.symbols("R")])
Rs = sp.symbols("R")
m1 = sp.simplify((u1.T * Mr * u1)[0, 0].subs(Rs, R_inertia))
print("m1 =", sp.simplify(m1))
print("   = I_xi I_zeta / I_z ?",
      sp.simplify(m1 - Ixi*Izeta/Mr[1, 1]) == 0)
print("   with p = I_xi/I_zeta:",
      sp.simplify(m1.subs(Ixi, p*Izeta) - p*Izeta/(p*sp.sin(al)**2 + sp.cos(al)**2)) == 0)

k1 = sp.simplify((u1.T * Kr * u1)[0, 0].subs(Rs, R_stiff))
w1sq = sp.simplify(k1 / m1.subs(Rs, R_inertia))
# express in the paper's form
w1_paper = (Kxi*(p*sp.sin(al)**2 + sp.cos(al)**2) /
            (Ixi*(q*sp.sin(be)**2 + sp.cos(be)**2)))
check = sp.simplify(sp.trigsimp(
    w1sq.subs({Ixi: p*Izeta, Kxi: q*Kzeta}) -
    w1_paper.subs({Ixi: p*Izeta, Kxi: q*Kzeta})))
print("\nomega_1^2 minus Eq.(27):", check)

m1 = I_xi*I_zeta/(I_xi*sin(alpha)**2 + I_zeta*cos(alpha)**2)
   = I_xi I_zeta / I_z ? True
   with p = I_xi/I_zeta: True



omega_1^2 minus Eq.(27): 0


## 7.5 · Eq. (29): the rocking axis

The mode shape is $[\phi,\psi]=[1,R]$, so the rotation vector points along
$(1,0,R)$ and the axis is inclined by $\delta$ with $\tan\delta=-R$ (the sign
following the same convention as $\alpha$). Substituting Eq. (22) gives

$$\tan\delta=\frac{(1-p)\tan\alpha}{p\tan^2\alpha+1},\qquad
\tan(\alpha-\delta)=p\tan\alpha,\qquad \tan(\beta-\delta)=q\tan\beta$$

In [6]:
# Work in t = tan(alpha) throughout, and use the angle-difference identity
#     tan(a - d) = (tan a - tan d) / (1 + tan a tan d)
tan_alpha = t
tan_delta = -(p - 1)*t / (p*t**2 + 1)          # = -R, from Eq. (22)

tan_a_minus_d = sp.simplify((tan_alpha - tan_delta) / (1 + tan_alpha*tan_delta))
print("tan(alpha - delta) =", sp.simplify(tan_a_minus_d))
print("equals p tan(alpha)?", sp.simplify(tan_a_minus_d - p*t) == 0)

# and the stiffness twin, Eq. (32)
tb = sp.symbols("t_b", positive=True)
tan_delta_b = -(q - 1)*tb / (q*tb**2 + 1)
tan_b_minus_d = sp.simplify((tb - tan_delta_b) / (1 + tb*tan_delta_b))
print("tan(beta - delta)  =", sp.simplify(tan_b_minus_d))
print("equals q tan(beta)?", sp.simplify(tan_b_minus_d - q*tb) == 0)

tan(alpha - delta) = p*t
equals p tan(alpha)? True
tan(beta - delta)  = q*t_b
equals q tan(beta)? True


## 7.6 · Everything at once, numerically

Take the reconstructed example engine, run the whole chain by hand, and compare
with `orthomount` at every step.

In [7]:
I_xi, I_eta, I_zeta = 0.95, 1.30, 1.60
alpha = D(30.0)
K_xi, K_eta, K_zeta = 5.30e4, 4.0e4, 2.45e4
pn, qn = I_xi/I_zeta, K_xi/K_zeta

# --- hand-built, using only what the primer derived ------------------------
def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1.0, 0], [-s, 0, c]])

Jn = rot_y(alpha) @ np.diag([I_xi, I_eta, I_zeta]) @ rot_y(alpha).T
Mr_n = Jn[np.ix_([0, 2], [0, 2])]
R_hand = -Mr_n[0, 1] / Mr_n[1, 1]

# solve Eq.(22) for beta:  R q tan^2(b) - (q-1) tan(b) + R = 0
aq, bq, cq = R_hand*qn, -(qn - 1.0), R_hand
disc = bq**2 - 4*aq*cq
t1, t2 = (-bq + np.sqrt(disc))/(2*aq), (-bq - np.sqrt(disc))/(2*aq)
beta_hand = np.arctan(min([t1, t2], key=abs))

Kbn = rot_y(beta_hand) @ np.diag([K_xi, K_eta, K_zeta]) @ rot_y(beta_hand).T
Kr_n = Kbn[np.ix_([0, 2], [0, 2])]

u1n = np.array([1.0, R_hand])
m1_hand = u1n @ Mr_n @ u1n
k1_hand = u1n @ Kr_n @ u1n
f1_hand = np.sqrt(k1_hand/m1_hand)/(2*np.pi)
delta_hand = np.arctan(-R_hand)

print("            hand-derived        orthomount")
print(f"R        {R_hand:16.8f}   {om.R_from_inertia(pn, alpha):14.8f}")
print(f"beta     {Dg(beta_hand):16.8f}   {Dg(om.solve_beta_for_R(R_hand, qn)[0]):14.8f}   [deg]")
print(f"m1       {m1_hand:16.8f}   {om.modal_mass_paper(I_xi, I_zeta, alpha):14.8f}")
print(f"f1       {f1_hand:16.8f}   "
      f"{om.omega1_paper(K_xi, I_xi, pn, qn, alpha, beta_hand)/2/np.pi:14.8f}   [Hz]")
print(f"delta    {Dg(delta_hand):16.8f}   {Dg(om.delta_paper(pn, alpha)):14.8f}   [deg]")

            hand-derived        orthomount
R             -0.19579705      -0.19579705
beta         -10.21125816     -10.21125816   [deg]
m1             1.05739130       1.05739130
f1            34.99800414      34.99800414   [Hz]
delta         11.07819699      11.07819699   [deg]


In [8]:
# --- and confirm the mode really is silent, in the full 6-DOF system -------
M6, K6 = om.paper_system(40.0, I_xi, I_eta, I_zeta, alpha,
                         K_xi, K_eta, K_zeta, beta_hand,
                         K_trans=(1.8e7, 3.5e6, 1.7e7))
res = om.modal_from_matrices(M6, K6)
F = np.array([0, 0, 0, 1.0, 0, 0])
part = res.normalised_participation(F)

print("mode   frequency      participation")
for fr, lab, pp in zip(res.f_hz, res.labels, part):
    print(f"  {fr:8.2f} Hz  {lab:<14s}  {pp:.3e}")
print(f"\nnumber of modes that respond at all: {int(np.sum(part > 1e-8))}")

mode   frequency      participation
     21.15 Hz  psi (yaw)       3.155e-17
     27.92 Hz  theta (pitch)   0.000e+00
     35.00 Hz  phi (roll)      9.814e-01
     47.08 Hz  y (lateral)     0.000e+00
    103.76 Hz  z (bounce)      0.000e+00
    106.76 Hz  x (fore/aft)    0.000e+00

number of modes that respond at all: 1


## 7.7 · Symbol map

Where each thing lives, so you can move between the paper, the primer and the
code without re-deriving anything.

| paper | meaning | primer | `orthomount` |
|---|---|---|---|
| Eq. (1) | $[M]\ddot q+[K]q=f$ | NB 1 | `MountSystem.M`, `.K` |
| Eq. (3) | eigenproblem | NB 2 | `MountSystem.modal()` |
| Eqs. (4)–(5) | $[M]$/$[K]$ orthogonality | NB 3 | verified in `test_orthomount.py` |
| Eq. (6) | modal superposition | NB 3–4 | `modal_response()` |
| $\{u^r\}^\mathsf{T}\{F\}$ | participation factor | NB 4 | `ModalResult.participation()` |
| Eq. (11) | $\{u^2\}=[0,1]^\mathsf{T}$ | NB 4.3 | — |
| Eqs. (13)–(16) | the condition on $R$ | NB 4.5 | `R_from_inertia`, `R_from_stiffness` |
| Eq. (20) | inertia tensor rotation | NB 5.4 | `inertia_tensor_from_principal` |
| Eq. (21) | stiffness rotation | NB 6 | `stiffness_block_from_principal` |
| Eq. (22) | $p,q,\alpha,\beta$ relation | NB 7.3 | `solve_beta_for_R` |
| Eq. (25) | modal inertia | NB 7.4 | `modal_mass_paper` |
| Eq. (27) | $\omega_1$ | NB 7.4 | `omega1_paper`, `K_xi_for_target_f1` |
| Eq. (29) | rocking axis $\delta$ | NB 7.5 | `delta_paper` |
| — | mount → $6\times6$ $[K]$ | NB 6.2 | `Mount.K6()` |
| — | elastic centre | NB 6.4 | `MountSystem.elastic_centre` |

---

## Where to go next

Back to the main walkthrough: **`../orthogonal_engine_mounts.ipynb`**, which
takes this machinery and applies it to the paper's own machine, adds real crank
excitation, damping and a mount-layout solver.